# Exercice 7 - Actifs Pondérés par le Risque de Crédit

$$RWA = LGD \times \left(\Phi\left(\frac{\Phi^{-1}(PD) + \sqrt{\rho}\, \Phi^{-1}(0.999)}{\sqrt{1-\rho}}\right) - PD\right) \times MA \times SF \times MCR \times EAD$$

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

## Question 1 - Définitions

**LGD** (Loss Given Default) : c'est la perte subie en cas de défaut, en % de l'exposition. Si on récupere 40% de la créance, LGD = 60%. C'est le complement du taux de recouvrement.

**PD** (Probability of Default) : probabilité que l'emprunteur fasse défaut dans l'année (horizon 1 an pour l'IRB). Estimée par la banque à partir de ses modeles de scoring / notations internes.

## Question 2 - Interprétation

Le terme central de la formule :
$$K = LGD \times \left(\Phi\left(\frac{\Phi^{-1}(PD) + \sqrt{\rho}\, \Phi^{-1}(0.999)}{\sqrt{1-\rho}}\right) - PD\right)$$

C'est le **capital économique unitaire**. Il correspond à :
- $\Phi(...)$ = la VaR 99.9% du taux de defaut conditionnel (formule de Vasicek, portefeuille granulaire)
- On soustrait PD = l'expected loss (les provisions couvrent cette partie)
- On multiplie par LGD pour avoir la perte en euros

En gros le capital doit couvrir la perte inattendue au quantile 99.9%. C'est directement issu du modele de Vasicek vu en cours.

In [ ]:
def K_irb(PD, LGD, rho):
    """Capital unitaire IRB"""
    var99 = norm.cdf((norm.ppf(PD) + np.sqrt(rho)*norm.ppf(0.999)) / np.sqrt(1-rho))
    return LGD * (var99 - PD)

## Question 3 - Corrélation réglementaire

ρ n'est pas estimé par la banque mais imposé par le regulateur. Il dépend du type d'exposition et de la PD.

In [ ]:
# formules réglementaires
def rho_corporate(PD):
    """Grandes entreprises (formule Bâle II)"""
    return 0.12 * (1-np.exp(-50*PD))/(1-np.exp(-50)) + 0.24 * (1 - (1-np.exp(-50*PD))/(1-np.exp(-50)))

def rho_retail_autre(PD):
    """Retail - autres expositions"""
    return 0.03 * (1-np.exp(-35*PD))/(1-np.exp(-35)) + 0.16 * (1 - (1-np.exp(-35*PD))/(1-np.exp(-35)))

In [ ]:
PD_range = np.linspace(0.0005, 0.15, 200)

plt.figure(figsize=(8, 5))
plt.plot(PD_range*100, [rho_corporate(p) for p in PD_range], 'b-', lw=2, label='Corporate')
plt.plot(PD_range*100, [rho_retail_autre(p) for p in PD_range], 'r-', lw=2, label='Retail autre')
plt.axhline(0.15, color='g', ls='--', alpha=0.6, label='Hypothécaire (fixe)')
plt.axhline(0.04, color='purple', ls='--', alpha=0.6, label='Renouvelable (fixe)')
plt.xlabel('PD (%)')
plt.ylabel('ρ')
plt.title('Corrélation réglementaire vs PD')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 3a. Etude de $f(x,y) = \frac{1-e^{-xy}}{1-e^{-y}}$

Cette fonction est le "poids" qui interpole entre les bornes de ρ.

In [ ]:
def f_xy(x, y):
    return (1 - np.exp(-x*y)) / (1 - np.exp(-y))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

x = np.linspace(0.001, 1, 100)
for yy in [10, 35, 50]:
    ax1.plot(x, f_xy(x, yy), label=f'y={yy}')
ax1.set_xlabel('x')
ax1.set_ylabel('f(x,y)')
ax1.set_title('f en fonction de x')
ax1.legend()
ax1.grid(alpha=0.3)

y = np.linspace(1, 80, 100)
for xx in [0.01, 0.05, 0.2]:
    ax2.plot(y, f_xy(xx, y), label=f'x={xx}')
ax2.set_xlabel('y')
ax2.set_ylabel('f(x,y)')
ax2.set_title('f en fonction de y')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 3b. Bornes de la corrélation

Comme $f(x,y) \in [0, 1]$ pour $x \geq 0$ :
- $f \to 0$ quand $x \to 0$ (PD très faible)
- $f \to 1$ quand $x \to \infty$ (PD élevée)

Pour les corporates : $\rho = 0.12 \cdot f + 0.24 \cdot (1-f)$ donc :
- PD → 0 : ρ → 0.24
- PD → ∞ : ρ → 0.12

**Bornes : ρ ∈ [0.12, 0.24]**

Pour retail : **ρ ∈ [0.03, 0.16]**

La corrélation decroît avec PD : les mauvais emprunteurs font défaut pour des raisons idiosyncratiques plutot que systémiques.

In [ ]:
# verification numerique des bornes
print(f"Corporate: rho(PD~0) = {rho_corporate(0.0001):.4f}, rho(PD=20%) = {rho_corporate(0.20):.4f}")
print(f"Retail:    rho(PD~0) = {rho_retail_autre(0.0001):.4f}, rho(PD=20%) = {rho_retail_autre(0.20):.4f}")

## Question 4 - Maturity Adjustment

$$MA = \frac{1 + (M - 2.5) \times b}{1 - 1.5 \times b}$$

avec $b = (0.11852 - 0.05478 \ln(PD))^2$

In [ ]:
def MA(M, PD):
    b = (0.11852 - 0.05478*np.log(PD))**2
    return (1 + (M - 2.5)*b) / (1 - 1.5*b)

In [ ]:
# 4a - MA vs M pour PD=0.2% et PD=4%
M_range = np.linspace(0.5, 5, 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(M_range, [MA(m, 0.002) for m in M_range], 'b-', lw=2, label='PD=0.2%')
ax1.plot(M_range, [MA(m, 0.04) for m in M_range], 'r-', lw=2, label='PD=4%')
ax1.axhline(1, color='grey', ls='--', alpha=0.5)
ax1.set_xlabel('M (années)')
ax1.set_ylabel('MA')
ax1.set_title('4a. MA vs Maturité')
ax1.legend()
ax1.grid(alpha=0.3)

# 4b - MA vs PD pour M=3
PD_plot = np.linspace(0.001, 0.05, 100)
ax2.plot(PD_plot*100, [MA(3, pd) for pd in PD_plot], 'b-', lw=2)
ax2.axhline(1, color='grey', ls='--', alpha=0.5)
ax2.set_xlabel('PD (%)')
ax2.set_ylabel('MA')
ax2.set_title('4b. MA vs PD (M=3)')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 4c. Origine de la formule

Le maturity adjustment corrige le modele Vasicek (qui est à horizon 1 an) pour les prêts de maturité differente :
- MA = 1 quand M = 1 an (on peut vérifier numériquement)
- MA > 1 pour M > 1 : les prêts longs sont pénalisés (risque de downgrade/migration)
- MA < 1 pour M < 1 : les prêts courts sont avantagés

Le parametre b croît quand PD diminue : les emprunteurs bien notés sont plus sensibles à la maturité car ils ont plus de "marge" pour migrer vers le défaut.

In [ ]:
# verification : MA(M=1) = 1
print(f"MA(M=1, PD=1%) = {MA(1, 0.01):.4f}")
print(f"MA(M=1, PD=5%) = {MA(1, 0.05):.4f}")
print("MA(M=1) = 1 quelle que soit PD (c'est la calibration du modele).")

## Question 5 - Scaling Factor = 1.06

Le SF de 1.06 a été introduit par le Comité de Bâle comme un coussin de sécurité. Son but est de s'assurer que le passage à l'approche IRB ne réduise pas le niveau global de fonds propres dans le système bancaire par rapport à Bâle I. C'est une marge conservatrice de +6% pour compenser les incertitudes du modèle.

## Question 6 - MCR = 12.5

Le MCR vaut $\frac{1}{8\%} = 12.5$

Ca vient directement du ratio de solvabilité de Bâle : Fonds Propres / RWA ≥ 8%.

Le facteur 12.5 sert à convertir le capital requis K en RWA :
- On veut que Fonds Propres ≥ K × EAD
- Mais la contrainte regulatoire s'exprime comme FP/RWA ≥ 8%
- Donc RWA = K × 12.5 × EAD, ce qui donne FP ≥ 8% × RWA = 8% × 12.5 × K × EAD = K × EAD

C'est juste une convention d'écriture pour rester cohérent avec le ratio de 8%.

In [ ]:
# Exemple de calcul complet
PD_ex = 0.02
LGD_ex = 0.45
EAD_ex = 1_000_000
M_ex = 3
SF = 1.06
MCR = 12.5

rho_ex = rho_corporate(PD_ex)
K_ex = K_irb(PD_ex, LGD_ex, rho_ex)
MA_ex = MA(M_ex, PD_ex)
RWA = K_ex * MA_ex * SF * MCR * EAD_ex

print(f"PD={PD_ex*100}%, LGD={LGD_ex*100}%, M={M_ex}ans, EAD={EAD_ex:,.0f}€")
print(f"rho = {rho_ex:.4f}")
print(f"K = {K_ex:.4f}")
print(f"MA = {MA_ex:.4f}")
print(f"RWA = {RWA:,.0f} €")
print(f"Capital requis = 8% x RWA = {0.08*RWA:,.0f} € = {0.08*RWA/EAD_ex*100:.1f}% de l'EAD")